# Q6 — 장기 학습(50 에폭) + TTA로 macro-F1 향상
Q3의 개선 설정(Focal Loss + class-balanced 가중치)을 그대로 쓰되, 두 가지를 추가한다.
1) **50 에폭 학습 + 조기종료**(검증 macro-F1 기준), 2) **TTA**(test-time augmentation): 추론 시 회전·반전한 8가지 버전의 예측을 평균.
목표: macro-F1 0.90 → 0.94 근처. 비교 기준 = Q3(0.899).

In [ ]:
# --- 한글 폰트 + import ---
import sys
from pathlib import Path
_p=Path.cwd()
for c in [_p,_p.parent,_p.parent.parent]:
    if (c/"utils").is_dir(): sys.path.insert(0,str(c)); PROJ=c; break
else: PROJ=_p
from utils.korean_font import set_korean_font
set_korean_font()
import numpy as np, matplotlib.pyplot as plt, time, itertools
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, recall_score

CLASSES=["Center","Donut","Edge-Loc","Edge-Ring","Loc","Near-full","Random","Scratch","none"]
NUM=len(CLASSES)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
PROC=PROJ/"data"/"processed"; CKPT=PROJ/"models"/"checkpoints"; CKPT.mkdir(parents=True,exist_ok=True)
FIG=PROJ/"results"/"figures"; FIG.mkdir(parents=True,exist_ok=True)
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- 설정 ---
EPOCHS=50; PATIENCE=8           # 조기종료: 8에폭 동안 val 개선 없으면 중단
BATCH=256; LR=3e-4
FOCAL_GAMMA=2.0; CB_BETA=0.999
torch.manual_seed(42); np.random.seed(42)
print(f"EPOCHS={EPOCHS} (조기종료 patience={PATIENCE}), Focal γ={FOCAL_GAMMA}, CB β={CB_BETA}")

In [ ]:
# --- 데이터 + Dataset ---
def load_split(n):
    d=np.load(PROC/f"wafer_{n}.npz",allow_pickle=True); return d["X"],d["y"]
Xtr,ytr=load_split("train"); Xva,yva=load_split("val"); Xte,yte=load_split("test")
counts=np.bincount(ytr,minlength=NUM)
print("train:",Xtr.shape,"| val:",Xva.shape,"| test:",Xte.shape)

class WaferDS(Dataset):
    def __init__(s,X,y,train=False): s.X=X; s.y=y; s.train=train
    def __len__(s): return len(s.X)
    def __getitem__(s,i):
        img=s.X[i].astype(np.float32)/2.0
        if s.train:
            k=np.random.randint(0,4)
            if k: img=np.rot90(img,k).copy()
            if np.random.rand()<0.5: img=np.fliplr(img).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy()
        img=(img-0.5)/0.5
        img=np.expand_dims(img,0).repeat(3,0)
        return torch.from_numpy(img), int(s.y[i])

mk=lambda X,y,tr: DataLoader(WaferDS(X,y,tr),batch_size=BATCH,shuffle=tr,num_workers=0,pin_memory=(DEVICE=="cuda"))
tr_loader=mk(Xtr,ytr,True); va_loader=mk(Xva,yva,False)

In [ ]:
# --- 모델 + 손실 (Q3와 동일 전략) ---
try: net=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
except Exception: net=models.resnet18(pretrained=True)
net.fc=nn.Linear(net.fc.in_features,NUM); net=net.to(DEVICE)

eff=1-np.power(CB_BETA,counts); cbw=(1-CB_BETA)/np.maximum(eff,1e-12)
cbw=(cbw/cbw.sum()*NUM).astype(np.float32); class_w=torch.tensor(cbw,device=DEVICE)

class FocalLoss(nn.Module):
    def __init__(s,weight=None,gamma=2.0): super().__init__(); s.w=weight; s.g=gamma
    def forward(s,logits,target):
        logp=F.log_softmax(logits,1); ce=F.nll_loss(logp,target,weight=s.w,reduction="none")
        pt=logp.exp().gather(1,target.unsqueeze(1)).squeeze(1)
        return ((1-pt)**s.g*ce).mean()

criterion=FocalLoss(class_w,FOCAL_GAMMA)
optimizer=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)

In [ ]:
# --- 50에폭 학습 + 조기종료 ---
@torch.no_grad()
def quick_eval(loader):
    net.eval(); ys=[];ps=[]
    for xb,yb in loader:
        ps.append(net(xb.to(DEVICE)).argmax(1).cpu().numpy()); ys.append(yb.numpy())
    y=np.concatenate(ys);p=np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p,average="macro")

best=-1; wait=0; best_path=CKPT/"q6_resnet18_50ep.pt"; hist={"loss":[],"val_f1":[]}
for ep in range(1,EPOCHS+1):
    net.train(); t0=time.time(); run=0.0
    for bi,(xb,yb) in enumerate(tr_loader):
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad(); loss=criterion(net(xb),yb); loss.backward(); optimizer.step(); run+=loss.item()
        if bi%100==0: print(f"  ep{ep} {bi}/{len(tr_loader)} loss={loss.item():.3f}",end="\r")
    scheduler.step()
    va_acc,va_f1=quick_eval(va_loader); hist["loss"].append(run/len(tr_loader)); hist["val_f1"].append(va_f1)
    flag=""
    if va_f1>best:
        best=va_f1; wait=0; torch.save({"model":net.state_dict(),"classes":CLASSES},best_path); flag="  <- best"
    else:
        wait+=1
    print(f"[ep{ep}/{EPOCHS}] loss={run/len(tr_loader):.3f} | val_macroF1={va_f1:.3f} | {time.time()-t0:.0f}s{flag}")
    if wait>=PATIENCE:
        print(f"\n조기종료: {PATIENCE}에폭 동안 개선 없음 (best val macro-F1={best:.4f})"); break
print("\nbest val macro-F1:", round(best,4))

In [ ]:
# --- 학습 곡선 ---
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(hist["loss"],marker="."); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(hist["val_f1"],marker="."); ax[1].set_title("val macro-F1"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(FIG/"q6_curves.png",dpi=110,bbox_inches="tight"); plt.show()

In [ ]:
# --- TTA 추론: 회전4 x 반전2 = 8가지 예측 평균 ---
net.load_state_dict(torch.load(best_path,map_location=DEVICE)["model"]); net.eval()

@torch.no_grad()
def predict(X, tta=True):
    transforms=[(k,f) for k in range(4) for f in [False,True]] if tta else [(0,False)]
    probs=np.zeros((len(X),NUM),dtype=np.float32)
    for i in range(0,len(X),512):
        base=X[i:i+512].astype(np.float32)/2.0           # (B,128,128)
        acc=None
        for k,f in transforms:
            arr=base
            if k: arr=np.rot90(arr,k,axes=(1,2))
            if f: arr=arr[:,:,::-1]
            arr=(np.ascontiguousarray(arr)-0.5)/0.5
            arr=np.expand_dims(arr,1).repeat(3,1)
            out=net(torch.from_numpy(arr).to(DEVICE)).softmax(1).cpu().numpy()
            acc=out if acc is None else acc+out
        probs[i:i+512]=acc/len(transforms)
    return probs.argmax(1)

p_plain=predict(Xte,tta=False)
p_tta  =predict(Xte,tta=True)
f1_plain=f1_score(yte,p_plain,average="macro"); acc_plain=accuracy_score(yte,p_plain)
f1_tta  =f1_score(yte,p_tta,average="macro");   acc_tta=accuracy_score(yte,p_tta)
print("="*46)
print(f"  Q3 기준                 macro-F1 = 0.899")
print(f"  Q6 50ep (TTA 없음)      macro-F1 = {f1_plain:.3f} | acc={acc_plain:.3f}")
print(f"  Q6 50ep + TTA           macro-F1 = {f1_tta:.3f} | acc={acc_tta:.3f}")
print("="*46)
print(f"  Q3 대비 향상: {f1_tta-0.899:+.3f}")

In [ ]:
# --- 최종(TTA) 클래스별 리포트 + 혼동행렬 ---
print(classification_report(yte,p_tta,target_names=CLASSES,digits=3,zero_division=0))
cm=confusion_matrix(yte,p_tta); cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
ax.set_xticks(range(NUM)); ax.set_xticklabels(CLASSES,rotation=45,ha="right")
ax.set_yticks(range(NUM)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("예측"); ax.set_ylabel("실제"); ax.set_title(f"Q6+TTA 혼동행렬 (macro-F1={f1_tta:.3f})")
for i,j in itertools.product(range(NUM),range(NUM)):
    ax.text(j,i,f"{cmn[i,j]:.2f}",ha="center",va="center",color="white" if cmn[i,j]>0.5 else "black",fontsize=7)
fig.colorbar(im,fraction=0.046); plt.tight_layout()
plt.savefig(FIG/"q6_confusion.png",dpi=110,bbox_inches="tight"); plt.show()

## 정리

- **50에폭 + 조기종료**로 충분히 수렴시키고, **TTA**로 추론 안정성을 높였다.
- TTA는 회전·반전한 8개 예측을 평균하므로 단일 예측보다 견고하다(웨이퍼맵은 회전 불변이라 특히 유효).
- 목표(macro-F1 0.94)에 도달했고 결과가 만족스러우면, 이 체크포인트(`q6_resnet18_50ep.pt`)와 수치를 `project/`·README·REPORT에 반영하면 된다.
- 더 올리고 싶으면: 소수 클래스 오버샘플링(WeightedRandomSampler), 더 큰 백본(ResNet34/EfficientNet-B0), 앙상블.